In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# importerer data
model_ready_df = pd.read_csv("../output/model_ready.csv")

In [2]:
# sjekker at model_ready data stemmer
model_ready_df.head()

,timestamp,hour,day_of_week,month,station,free_bikes,free_bikes_next_hour,arrivals,departures,trips_count,temperature,precipitation,wind_speed
0,2024-04-10 23:00:00+02:00,23,2,4,Akvariet,0,0.0,3,3,14,5.1,0,17.4
1,2024-04-10 23:00:00+02:00,23,2,4,Dreggsallmenningen Sør,0,12.0,1,1,14,5.1,0,17.4
2,2024-04-10 23:00:00+02:00,23,2,4,Studentboligene,0,15.0,1,1,14,5.1,0,17.4
3,2024-04-10 23:00:00+02:00,23,2,4,Florida Bybanestopp,0,0.0,1,1,14,5.1,0,17.4
4,2024-04-10 23:00:00+02:00,23,2,4,Møllendalsplass,0,4.0,1,1,14,5.1,0,17.4


In [3]:
# deler data i mål- og prediktorvariabler og gjør om stations til å bruke one-hot encode
X = pd.get_dummies(model_ready_df.drop(columns=["free_bikes_next_hour", "timestamp"]),
                   columns=["station"], prefix="station")
y = model_ready_df["free_bikes_next_hour"]

In [4]:
# sjekker at X og y stemmer overens med data
X.head()

,hour,day_of_week,month,free_bikes,arrivals,departures,trips_count,temperature,precipitation,wind_speed,station_Akvariet,station_Damsgårdsveien 71,station_Dreggsallmenningen Sør,station_Florida Bybanestopp,station_Grieghallen,station_Høyteknologisenteret,station_Møllendalsplass,station_Studentboligene,station_Torgallmenningen
0,23,2,4,0,3,3,14,5.1,0,17.4,True,False,False,False,False,False,False,False,False
1,23,2,4,0,1,1,14,5.1,0,17.4,False,False,True,False,False,False,False,False,False
2,23,2,4,0,1,1,14,5.1,0,17.4,False,False,False,False,False,False,False,True,False
3,23,2,4,0,1,1,14,5.1,0,17.4,False,False,False,True,False,False,False,False,False
4,23,2,4,0,1,1,14,5.1,0,17.4,False,False,False,False,False,False,True,False,False


In [5]:
y.head()

0     0.0
1    12.0
2    15.0
3     0.0
4     4.0
Name: free_bikes_next_hour, dtype: float64

In [6]:
# deler opp i trenings-, validerings- og testdata
X_train, X_valtest, y_train, y_valtest = train_test_split(X, y, train_size=0.7, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_valtest, y_valtest, train_size=0.5, shuffle=False)

In [7]:
# trener en grunnlinjemodell på treningsdata og finner RMSE på all data
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_train_rmse = root_mean_squared_error(y_train, baseline.predict(X_train))
baseline_val_rmse = root_mean_squared_error(y_val, baseline.predict(X_val))
baseline_test_rmse = root_mean_squared_error(y_test, baseline.predict(X_test))

print("Resultat for baseline-modell:\n")
print("Den beste treningsdata-RMSE-en er:", round(baseline_train_rmse, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(baseline_val_rmse, 3),
      "\nDen beste testdata-RMSE-en er:", round(baseline_test_rmse, 3))

Resultat for baseline-modell:

Den beste treningsdata-RMSE-en er: 7.1 
Den beste valideringsdata-RMSE-en er: 5.059 
Den beste testdata-RMSE-en er: 8.837


In [8]:
# trener en lineær regresjonsmodell på treningsdata og finner RMSE på all data
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
train_rmse_linreg_scores = root_mean_squared_error(y_train, lin_reg.predict(X_train))
val_rmse_linreg_scores = root_mean_squared_error(y_val, lin_reg.predict(X_val))
test_rmse_linreg_scores = root_mean_squared_error(y_test, lin_reg.predict(X_test))

print("Resultat for lineær regresjon:\n")
print("Den beste treningsdata-RMSE-en er:", round(train_rmse_linreg_scores, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(val_rmse_linreg_scores, 3),
      "\nDen beste testdata-RMSE-en er:", round(test_rmse_linreg_scores, 3))

Resultat for lineær regresjon:

Den beste treningsdata-RMSE-en er: 1.38 
Den beste valideringsdata-RMSE-en er: 1.094 
Den beste testdata-RMSE-en er: 1.77


In [9]:
# trener lasso-modell på ulike verdier av alpha
lasso_models = {alpha: Lasso(alpha=alpha) for alpha in np.arange(0.01, 10, 0.1)}
train_rmse_scores = {}
val_rmse_scores = {}
test_rmse_scores = {}

for alpha, model in lasso_models.items():
    model.fit(X_train, y_train)
    train_rmse_scores[alpha] = root_mean_squared_error(y_train, model.predict(X_train))
    val_rmse_scores[alpha] = root_mean_squared_error(y_val, model.predict(X_val))
    test_rmse_scores[alpha] = root_mean_squared_error(y_test, model.predict(X_test))

In [10]:
# finner hvilken alpha som gir laveste RMSE
best_alpha = min(val_rmse_scores, key=val_rmse_scores.get)

# finner RMSE på all data for denne alpha-verdien
best_alpha_train_rmse = train_rmse_scores[best_alpha]
best_alpha_val_rmse = val_rmse_scores[best_alpha]
best_alpha_test_rmse = test_rmse_scores[best_alpha]

print("Resultat for lasso-regresjon:\n")
print("Den beste alpha-en er", round(best_alpha, 3))
print("Den beste treningsdata-RMSE-en er:", round(best_alpha_train_rmse, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(best_alpha_val_rmse, 3),
      "\nDen beste testdata-RMSE-en er:", round(best_alpha_test_rmse, 3))

Resultat for lasso-regresjon:

Den beste alpha-en er 0.41
Den beste treningsdata-RMSE-en er: 1.486 
Den beste valideringsdata-RMSE-en er: 1.033 
Den beste testdata-RMSE-en er: 1.671


In [11]:
# lager datasett med polynomielle data og utfører samme prosedyre med alpha
poly = PolynomialFeatures(degree=2)
X_train_pf = poly.fit_transform(X_train)
X_val_pf = poly.transform(X_val)
X_test_pf = poly.transform(X_test)

pf_models = {alpha: Lasso(alpha=alpha) for alpha in np.arange(0.01, 10, 0.1)}

train_rmse_scores_pf = {}
val_rmse_scores_pf = {}
test_rmse_scores_pf = {}

for alpha, model in pf_models.items():
    model.fit(X_train_pf, y_train)
    train_rmse_scores_pf[alpha] = root_mean_squared_error(y_train, model.predict(X_train_pf))
    val_rmse_scores_pf[alpha] = root_mean_squared_error(y_val, model.predict(X_val_pf))
    test_rmse_scores_pf[alpha] = root_mean_squared_error(y_test, model.predict(X_test_pf))

best_alpha_pf = min(val_rmse_scores_pf, key=val_rmse_scores_pf.get)

best_alpha_train_rmse_pf = train_rmse_scores_pf[best_alpha_pf]
best_alpha_val_rmse_pf = val_rmse_scores_pf[best_alpha_pf]
best_alpha_test_rmse_pf = test_rmse_scores_pf[best_alpha_pf]

print("Resultat for lasso-regresjon med polynom av andre grad:\n")
print("Den beste alpha-en er", best_alpha_pf)
print("Den beste treningsdata-RMSE-en er:", round(best_alpha_train_rmse_pf, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(best_alpha_val_rmse_pf, 3),
      "\nDen beste testdata-RMSE-en er:", round(best_alpha_test_rmse_pf, 3))

Resultat for lasso-regresjon med polynom av andre grad:

Den beste alpha-en er 0.01
Den beste treningsdata-RMSE-en er: 1.283 
Den beste valideringsdata-RMSE-en er: 1.031 
Den beste testdata-RMSE-en er: 1.784


In [12]:
# prøver en k-nærmeste nabo modell for noen verdier av k
k_neighbors = {n_neighbors: KNeighborsRegressor(n_neighbors=n_neighbors) for n_neighbors in [1, 5, 10, 20, 30]}

train_rmse_scores_k_neighbors = {}
val_rmse_scores_k_neighbors = {}
test_rmse_scores_k_neighbors = {}

for neighbor, model in k_neighbors.items():
    model.fit(X_train, y_train)
    train_rmse_scores_k_neighbors[neighbor] = root_mean_squared_error(y_train, model.predict(X_train))
    val_rmse_scores_k_neighbors[neighbor] = root_mean_squared_error(y_val, model.predict(X_val))
    test_rmse_scores_k_neighbors[neighbor] = root_mean_squared_error(y_test, model.predict(X_test))

best_neighborcount = min(val_rmse_scores_k_neighbors, key=val_rmse_scores_k_neighbors.get)

best_train_rmse_neighborcount = train_rmse_scores_k_neighbors[best_neighborcount]
best_val_rmse_neighborcount = val_rmse_scores_k_neighbors[best_neighborcount]
best_test_rmse_neighborcount = test_rmse_scores_k_neighbors[best_neighborcount]

print("Resultat for k-nærmeste-nabo-regresjon:\n")
print("Det beste antall naboer er:", best_neighborcount)
print("Den beste treningsdata-RMSE-en er:", round(best_train_rmse_neighborcount, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(best_val_rmse_neighborcount, 3),
      "\nDen beste testdata-RMSE-en er:", round(best_test_rmse_neighborcount, 3))
    

Resultat for k-nærmeste-nabo-regresjon:

Det beste antall naboer er: 5
Den beste treningsdata-RMSE-en er: 1.384 
Den beste valideringsdata-RMSE-en er: 1.305 
Den beste testdata-RMSE-en er: 2.175


In [13]:
# prøver noen flere modeller (unngår GaussianProcessRegressor grunnet filstørrelse)
models = {"Elastic Net": ElasticNet(random_state=0),
          "Random Forest": RandomForestRegressor(random_state=0),
          "Support Vector": make_pipeline(StandardScaler(), SVR(kernel="linear")),
          "Ridge": Ridge()}

for _, model in models.items():
    model.fit(X_train, y_train)

# sjekker RMSE på all data
curr_train_rmse = 0
curr_val_rmse = 0
curr_test_rmse = 0

best_train_rmse = 10
best_val_rmse = 10
best_test_rmse = 10

best_train_rmse_model = ""
best_val_rmse_model = ""
best_overall_in_models = ""

for name, model in models.items():
    curr_train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
    curr_val_rmse = root_mean_squared_error(y_val, model.predict(X_val))
    curr_test_rmse = root_mean_squared_error(y_test, model.predict(X_test))

    if (curr_train_rmse < best_train_rmse):
        best_train_rmse = curr_train_rmse
        best_train_rmse_model = name

    if (curr_val_rmse < best_val_rmse):
        best_val_rmse = curr_val_rmse
        best_val_rmse_model = name

    if (curr_test_rmse < best_test_rmse):
        best_test_rmse = curr_test_rmse
        best_overall_in_models = name


print("Den beste treningsmodellen er:", best_train_rmse_model)
print("Den beste valideringsmodellen er:", best_val_rmse_model)
print("Den beste test- og dermed også totalt beste modellen er:", best_overall_in_models)

Den beste treningsmodellen er: Random Forest
Den beste valideringsmodellen er: Support Vector
Den beste test- og dermed også totalt beste modellen er: Support Vector


In [14]:
train_rmse_models_scores = root_mean_squared_error(y_train, models[best_overall_in_models].predict(X_train))
val_rmse_models_scores = root_mean_squared_error(y_val, models[best_overall_in_models].predict(X_val))
test_rmse_models_scores = root_mean_squared_error(y_test, models[best_overall_in_models].predict(X_test))

print("Resultat for beste av en rekke modeller:", best_overall_in_models,"\n")
print("Den beste treningsdata-RMSE-en er:", round(train_rmse_models_scores, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(val_rmse_models_scores, 3),
      "\nDen beste testdata-RMSE-en er:", round(test_rmse_models_scores, 3))

Resultat for beste av en rekke modeller: Support Vector 

Den beste treningsdata-RMSE-en er: 1.509 
Den beste valideringsdata-RMSE-en er: 1.036 
Den beste testdata-RMSE-en er: 1.666


In [15]:
# finner modellen med best test-RMSE ut ifra tabellen
results = pd.DataFrame([
    {"model": "Baseline", "train_rmse": baseline_train_rmse, "val_rmse": baseline_val_rmse, "test_rmse": baseline_test_rmse},
    {"model": "LinearRegression", "train_rmse": train_rmse_linreg_scores, "val_rmse": val_rmse_linreg_scores, "test_rmse": test_rmse_linreg_scores},
    {"model": "Lasso", "train_rmse": best_alpha_train_rmse, "val_rmse": best_alpha_val_rmse, "test_rmse": best_alpha_test_rmse},
    {"model": "Poly + Lasso", "train_rmse": best_alpha_train_rmse_pf, "val_rmse": best_alpha_val_rmse_pf, "test_rmse": best_alpha_test_rmse_pf},
    {"model": "KNN", "train_rmse": best_train_rmse_neighborcount, "val_rmse": best_val_rmse_neighborcount, "test_rmse": best_test_rmse_neighborcount},
    {"model": best_overall_in_models, "train_rmse": train_rmse_models_scores, "val_rmse": val_rmse_models_scores,"test_rmse": test_rmse_models_scores}
])
print(results)
print("\nModellen med best test-RMSE er:", results.loc[results["test_rmse"].idxmin(), "model"],
      "med en test-RMSE på:", round(min(results["test_rmse"]), 3))

              model  train_rmse  val_rmse  test_rmse
0          Baseline    7.099952  5.059483   8.837275
1  LinearRegression    1.379526  1.094110   1.769914
2             Lasso    1.485501  1.033252   1.671329
3      Poly + Lasso    1.283069  1.031178   1.784392
4               KNN    1.383652  1.304938   2.175295
5    Support Vector    1.509478  1.035738   1.666400

Modellen med best test-RMSE er: Support Vector med en test-RMSE på: 1.666
